# 04_quantization_comparison: Symmetric vs. Asymmetric Quantization

This notebook demonstrates Uniform Symmetric vs. Uniform Asymmetric quantization. We write quantization scaling calculations from scratch in PyTorch, quantize model weights, and compute reconstruction error metrics.

### Formulations
1. **Symmetric Quantization**:
   $$s = \frac{\max(|X|)}{q_{\text{max}}}, \quad z = 0$$
2. **Asymmetric Quantization**:
   $$s = \frac{\max(X) - \min(X)}{q_{\text{max}} - q_{\text{min}}}, \quad z = \text{round}\left(\frac{-\min(X)}{s}\right) + q_{\text{min}}$$
3. **Quantized Mapping**:
   $$q = \text{clamp}\left(\text{round}\left(\frac{r}{s}\right) + z, q_{\text{min}}, q_{\text{max}}\right)$$
4. **De-quantization**:
   $$\hat{r} = s \cdot (q - z)$$

In [1]:
import torch

def quantize_symmetric(x, bits=8):
    q_max = (2 ** (bits - 1)) - 1
    scale = torch.max(torch.abs(x)) / q_max
    q = torch.clamp(torch.round(x / scale), -q_max, q_max)
    dequant = q * scale
    return q, dequant, scale

def quantize_asymmetric(x, bits=8):
    q_min = 0
    q_max = (2 ** bits) - 1
    x_min = torch.min(x)
    x_max = torch.max(x)
    
    scale = (x_max - x_min) / (q_max - q_min)
    zero_point = torch.round(-x_min / scale) + q_min
    zero_point = torch.clamp(zero_point, q_min, q_max)
    
    q = torch.clamp(torch.round(x / scale) + zero_point, q_min, q_max)
    dequant = scale * (q - zero_point)
    return q, dequant, scale, zero_point

In [2]:
# Test quantization on a weight matrix with dynamic outliers
torch.manual_seed(42)
weights = torch.randn(100, 100) * 2.0
# Add dynamic outliers representing emergent channel behaviors
weights[0, :] *= 10.0

print("Original weights sample:", weights[0, :5].tolist())

# Apply Symmetric INT8
q_sym, deq_sym, scale_sym = quantize_symmetric(weights, bits=8)
mse_sym = torch.mean((weights - deq_sym) ** 2).item()

# Apply Asymmetric INT8
q_asym, deq_asym, scale_asym, zp_asym = quantize_asymmetric(weights, bits=8)
mse_asym = torch.mean((weights - deq_asym) ** 2).item()

print(f"Symmetric Scale: {scale_sym.item():.4f}, Zero Point: 0")
print(f"Symmetric Quantization Reconstruction MSE Error:  {mse_sym:.6f}")
print(f"Asymmetric Scale: {scale_asym.item():.4f}, Zero Point: {zp_asym.item()}")
print(f"Asymmetric Quantization Reconstruction MSE Error: {mse_asym:.6f}")

Original weights sample: [38.538307189941406, 29.745681762695312, 18.01434326171875, -42.11042022705078, 13.568367958068848]
Symmetric Scale: 0.3952, Zero Point: 0
Symmetric Quantization Reconstruction MSE Error:  0.013146
Asymmetric Scale: 0.3708, Zero Point: 135.0
Asymmetric Quantization Reconstruction MSE Error: 0.011423


### Output Explanation & Verification

- **Scale & Zero Points**: Symmetric scale (**0.3952**, zero-point **0**) mapped the maximum absolute outlier amplitude. Asymmetric scale (**0.3708**, zero-point **135.0**) adjusted boundaries dynamically, mapping the minimum weight bound to the zero point.
- **Error Metric Verification**: The asymmetric quantization MSE error (**0.011423**) is smaller than the symmetric error (**0.013146**). This confirms that asymmetric ranges accommodate skewed distributions more accurately, validating the numerical precision benefit of dynamic scaling.